
# HTM — Final Pipeline (v4.1, No MLflow; Discrete Years + Log-AAV + Quantiles)

**What this version adds vs v4**
- Separate heads: **classifier for contract years (1–12)**, **regressor for AAV (log scale)**.
- **Quantile bands** for AAV (P10 / P50 / P90) + coverage check.
- **Tiered AAV metrics** (e.g., <$5M, $5–15M, $15–30M, $30M+).
- Clear **metrics tables** and **feature importances** (aggregated to raw features).
- Same data merge logic with **URL_ID** priority, and artifacts saved locally (no MLflow).


## 0) Setup & Imports

In [1]:

# !pip install scikit-learn pandas numpy joblib

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 200)


## 1) Data loading

In [2]:

from urllib.parse import quote

# --- Update tab names if needed ---
TAB_2025 = "Sheet1"   # e.g., "Hitters_2025"
TAB_2026 = "Sheet1"   # e.g., "Hitters_2026"
TAB_CON  = "Sheet1"   # e.g., "Contracts"

ID_2025 = "1RkF5ebxs4ysIAfuGLToGZ0y-q3th2GCI"
ID_2026 = "19LDvcmzHjW9YPW6rEelYiyht1ftuhCoR"
ID_CON  = "1Ohdgb12prXSMES4-HEV_X5-uFmXuIRuB"

def gsheet_to_csv_url(sheet_id: str, tab_name: str) -> str:
    return f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={quote(tab_name)}"

url_2025 = gsheet_to_csv_url(ID_2025, TAB_2025)
url_2026 = gsheet_to_csv_url(ID_2026, TAB_2026)
url_con  = gsheet_to_csv_url(ID_CON,  TAB_CON)

df_2025 = pd.read_csv(url_2025)
df_2026 = pd.read_csv(url_2026)
df_con  = pd.read_csv(url_con)

display({
    "2025_shape": df_2025.shape,
    "2026_shape": df_2026.shape,
    "contracts_shape": df_con.shape
})
df_2025.head(3)


{'2025_shape': (665, 59),
 '2026_shape': (1084, 64),
 'contracts_shape': (233, 7)}

,player,URL_ID,season,Age,Tm,Lg,WAR,G,PA,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,BA,OBP,SLG,OPS,OPS+,rOBA,Rbat+,TB,GIDP,HBP,SH,SF,IBB,Pos,Awards,Batting|rOBA,Batting|Rbat+,Batting|BAbip,Batting|ISO,Batting Ratios|HR%,Batting Ratios|SO%,Batting Ratios|BB%,Batted Ball|EV,Batted Ball|HardH%,Batted Ball|LD%,Batted Ball|GB%,Batted Ball|FB%,Batted Ball|GB/FB,Batted Ball|Pull%,Batted Ball|Cent%,Batted Ball|Oppo%,Win Probability|WPA,Win Probability|cWPA,Win Probability|RE24,Baserunning|RS%,Baserunning|SB%,Baserunning|XBT%,Unnamed: 27_level_0|Pos,Unnamed: 28_level_0|Awards
0,Juan Soto,https://www.baseball-reference.com/players/s/s...,2018,19,WSN,NL,3.0,116,494,414,77,121,25,1,22,70,5,2,79,99,0.292,0.406,0.517,0.923,142,0.396,143,214,9,0,1,0,10,*7/H,ROY-2,0.396,143,0.338,0.225,4.5,20.0,16.0,90.3,46.6,21.2,52.5,19.0,1.16,33.2,49.4,17.4,3.4,0.012,29.61,30.9,71.4,51.7,*7/H,ROY-2
1,Juan Soto,https://www.baseball-reference.com/players/s/s...,2019,20,WSN,NL,5.1,150,659,542,110,153,32,5,34,110,12,1,108,132,0.282,0.401,0.548,0.949,142,0.404,145,297,11,3,0,6,3,*7,MVP-9,0.404,145,0.312,0.266,5.2,20.0,16.4,92.0,51.3,26.7,41.6,26.9,0.73,32.4,48.2,19.4,3.6,0.021,38.72,33.0,92.3,40.7,*7,MVP-9
2,Juan Soto,https://www.baseball-reference.com/players/s/s...,2020,21,WSN,NL,2.3,47,196,154,39,54,14,0,13,37,6,2,41,28,0.351,0.490,0.695,1.185,217,0.494,221,107,1,1,0,0,12,*7/9D,"MVP-5,SS",0.494,221,0.363,0.344,6.6,14.3,20.9,92.1,51.6,28.6,53.2,15.9,1.14,21.4,58.7,19.8,2.1,0.008,31.86,31.3,75.0,25.0,*7/9D,"MVP-5,SS"


## 2) Helper functions: filter hitters & build five-season features (keeps URL_ID)

In [3]:

def filter_hitters(df):
    cols = df.columns.str.lower()
    if "is_hitter" in cols:
        return df[df.loc[:, cols=="is_hitter"].iloc[:,0].astype(bool)]
    if "is_pitcher" in cols:
        return df[~df.loc[:, cols=="is_pitcher"].iloc[:,0].astype(bool)]
    if "position" in cols or "primary_position" in cols:
        pos_col = "position" if "position" in cols else "primary_position"
        pos_col = df.columns[cols == pos_col][0]
        return df[~df[pos_col].astype(str).str.upper().str.contains("(^P$)|( P$)|( P,)|(^P,)", regex=True)]
    return df

NUM_COL_CANDIDATES = ["PA","WAR","wRC_plus","OBP","SLG","ISO","BB_pct","K_pct","wOBA","BsR","IL_days","AGE"]
CAT_COL_CANDIDATES = ["bats","primary_position","position","team"]
ID_COL_KEEP = ["URL_ID","PLAYER_ID","MLBID","ID","PLAYER_NAME","NAME"]

def _detect_long_format(df):
    cols = set(df.columns.str.lower())
    return ("season" in cols) or ("year" in cols)

def _melt_wide(df, season_range):
    import re
    base_cols = [c for c in ID_COL_KEEP if c in df.columns]
    long_frames = []
    for col in df.columns:
        m = re.match(r"(.+?)_(20\d{2})$", col)
        if m and int(m.group(2)) in season_range:
            base, yr = m.group(1), int(m.group(2))
            keep = base_cols + [col]
            tmp = df[keep].copy()
            tmp["SEASON"] = yr
            tmp.rename(columns={col: base}, inplace=True)
            long_frames.append(tmp)
    if not long_frames:
        raise ValueError("Could not detect wide season-suffixed columns.")
    long_df = long_frames[0]
    for fr in long_frames[1:]:
        long_df = long_df.merge(fr, on=base_cols + ["SEASON"], how="outer")
    return long_df

def build_5yr_features(hitters_df, window_end, seasons_back=5):
    season_range = list(range(window_end - seasons_back + 1, window_end + 1))
    df = hitters_df.copy()
    df.columns = [c.strip() for c in df.columns]
    df.rename(columns={c: c.upper() for c in df.columns}, inplace=True)

    if "PLAYER_ID" not in df.columns: 
        if "MLBID" in df.columns:
            df["PLAYER_ID"] = df["MLBID"]
        else:
            df["PLAYER_ID"] = np.arange(len(df))
    if "PLAYER_NAME" not in df.columns: 
        for name_col in ["NAME","PLAYER","PLAYERNAME"]:
            if name_col in df.columns:
                df["PLAYER_NAME"] = df[name_col]
                break
        else:
            df["PLAYER_NAME"] = df["PLAYER_ID"].astype(str)

    if _detect_long_format(df.rename(columns=str.lower)):
        season_col = "SEASON" if "SEASON" in df.columns else "YEAR"
        long_df = df[df[season_col].astype(int).isin(season_range)].copy()
        long_df.rename(columns={season_col: "SEASON"}, inplace=True)
    else:
        long_df = _melt_wide(df.rename(columns=str), season_range)
        long_df.columns = [c.upper() for c in long_df.columns]

    num_cols = [c for c in NUM_COL_CANDIDATES if c in long_df.columns]
    cat_cols = [c for c in CAT_COL_CANDIDATES if c.upper() in long_df.columns]
    cat_cols = [c.upper() for c in cat_cols]

    records = []
    for pid, g in long_df.groupby("PLAYER_ID"):
        rec = {"PLAYER_ID": pid}
        g_sorted = g.sort_values("SEASON")
        if "URL_ID" in g_sorted.columns and g_sorted["URL_ID"].notna().any():
            rec["URL_ID"] = g_sorted["URL_ID"].dropna().iloc[-1]
        if "PLAYER_NAME" in g_sorted.columns and g_sorted["PLAYER_NAME"].notna().any():
            rec["PLAYER_NAME"] = g_sorted["PLAYER_NAME"].dropna().iloc[-1]
        elif "NAME" in g_sorted.columns and g_sorted["NAME"].notna().any():
            rec["PLAYER_NAME"] = g_sorted["NAME"].dropna().iloc[-1]
        xs = g_sorted["SEASON"].values
        for col in num_cols:
            arr = g_sorted[col].astype(float).values
            rec[f"{col}_MEAN5"] = np.nanmean(arr) if arr.size else np.nan
            rec[f"{col}_LAST"]  = arr[-1] if arr.size else np.nan
            rec[f"{col}_STD5"]  = np.nanstd(arr, ddof=0) if arr.size else np.nan
            if arr.size > 1 and np.isfinite(arr).sum() > 1:
                x = xs[np.isfinite(arr)].astype(float); y = arr[np.isfinite(arr)].astype(float)
                rec[f"{col}_TREND"] = np.cov(x, y, ddof=0)[0,1] / (np.var(x, ddof=0) + 1e-12)
            else:
                rec[f"{col}_TREND"] = np.nan
        for col in cat_cols:
            rec[col] = g_sorted[col].dropna().iloc[-1] if (col in g_sorted.columns and g_sorted[col].notna().any()) else np.nan
        records.append(rec)
    feats = pd.DataFrame.from_records(records)
    return feats


## 3) Build five-season features and 2025 labels (URL_ID-based merge)

In [5]:

# Focus on hitters only
h25 = filter_hitters(df_2025).copy()
h26 = filter_hitters(df_2026).copy()

# Build features
X_2025 = build_5yr_features(h25, window_end=2024, seasons_back=5)  # uses 2020–2024
X_2026 = build_5yr_features(h26, window_end=2025, seasons_back=5)  # uses 2021–2025

# Prepare 2025 labels (contracts)
lab = df_con.copy()
lab.columns = [c.upper() for c in lab.columns]
lab_hit = filter_hitters(lab)

# Normalize key columns
if "CONTRACT_TOTAL" not in lab_hit.columns and "TOTAL_VALUE" in lab_hit.columns:
    lab_hit["CONTRACT_TOTAL"] = lab_hit["TOTAL_VALUE"]
if "CONTRACT_YEARS" not in lab_hit.columns and "YEARS" in lab_hit.columns:
    lab_hit["CONTRACT_YEARS"] = lab_hit["YEARS"]
if "AAV" not in lab_hit.columns and {"CONTRACT_TOTAL","CONTRACT_YEARS"}.issubset(lab_hit.columns):
    lab_hit["AAV"] = lab_hit["CONTRACT_TOTAL"] / lab_hit["CONTRACT_YEARS"]

# Filter to 2025 class only
off_col = "SIGNING_OFFSEASON" if "SIGNING_OFFSEASON" in lab_hit.columns else ("OFFSEASON" if "OFFSEASON" in lab_hit.columns else None)
if off_col is None:
    raise ValueError("Contracts sheet must contain SIGNING_OFFSEASON or OFFSEASON column.")
con_2025 = lab_hit[lab_hit[off_col].astype(str) == "2025"].copy()

# Merge key priority: URL_ID -> PLAYER_ID -> NAME
if "URL_ID" in con_2025.columns and "URL_ID" in X_2025.columns:
    MERGE_KEY = "URL_ID"
elif "PLAYER_ID" in con_2025.columns and "PLAYER_ID" in X_2025.columns:
    MERGE_KEY = "PLAYER_ID"
else:
    for n in ["PLAYER_NAME","NAME","PLAYER"]:
        if n in con_2025.columns:
            if "PLAYER_NAME" in X_2025.columns:
                con_2025 = con_2025.rename(columns={n: "PLAYER_NAME"})
                MERGE_KEY = "PLAYER_NAME"
            elif "NAME" in X_2025.columns:
                con_2025 = con_2025.rename(columns={n: "NAME"})
                MERGE_KEY = "NAME"
            break
    if 'MERGE_KEY' not in locals():
        raise ValueError("Could not find a common key (URL_ID/PLAYER_ID/NAME) to merge labels.")

y_df = con_2025[[MERGE_KEY, "CONTRACT_YEARS","CONTRACT_TOTAL","AAV"]].copy()
train_df = X_2025.merge(y_df, on=MERGE_KEY, how="inner")

print("Using merge key:", MERGE_KEY)
print("Train rows:", train_df.shape[0])
display(train_df.head(3))


Using merge key: URL_ID
Train rows: 433


,PLAYER_ID,URL_ID,PLAYER_NAME,PA_MEAN5,PA_LAST,PA_STD5,PA_TREND,WAR_MEAN5,WAR_LAST,WAR_STD5,WAR_TREND,OBP_MEAN5,OBP_LAST,OBP_STD5,OBP_TREND,SLG_MEAN5,SLG_LAST,SLG_STD5,SLG_TREND,AGE_MEAN5,AGE_LAST,AGE_STD5,AGE_TREND,CONTRACT_YEARS,CONTRACT_TOTAL,AAV
0,2,https://www.baseball-reference.com/players/s/s...,Juan Soto,196.0,196.0,0.0,NaN,2.3,2.3,0.0,NaN,0.490,0.490,0.0,NaN,0.695,0.695,0.0,NaN,21.0,21.0,0.0,NaN,15.0,765000000.0,51000000.0
1,3,https://www.baseball-reference.com/players/s/s...,Juan Soto,654.0,654.0,0.0,NaN,7.3,7.3,0.0,NaN,0.465,0.465,0.0,NaN,0.534,0.534,0.0,NaN,22.0,22.0,0.0,NaN,15.0,765000000.0,51000000.0
2,4,https://www.baseball-reference.com/players/s/s...,Juan Soto,664.0,664.0,0.0,NaN,5.5,5.5,0.0,NaN,0.401,0.401,0.0,NaN,0.452,0.452,0.0,NaN,23.0,23.0,0.0,NaN,15.0,765000000.0,51000000.0


## 4) ColumnTransformer: impute/scale numerics + encode categoricals

In [6]:

ID_COLS = {"PLAYER_ID","PLAYER_NAME","NAME","URL_ID"}
TARGET_COLS = {"CONTRACT_YEARS","CONTRACT_TOTAL","AAV"}
feature_cols = [c for c in train_df.columns if c not in ID_COLS | TARGET_COLS]

X = train_df[feature_cols].copy()
y_years = train_df["CONTRACT_YEARS"].astype(int).clip(1, 12)
y_aav   = train_df["AAV"].astype(float)

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
], remainder="drop")

print(f"Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}")


Numeric cols: 20, Categorical cols: 0


## 5) Contract Years — Classifier (RandomForest)

In [7]:

X_tr_y, X_val_y, y_tr_y, y_val_y = train_test_split(
    X, y_years, test_size=0.25, random_state=RANDOM_STATE, stratify=y_years
)

years_clf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=800, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
])
years_clf.fit(X_tr_y, y_tr_y)
pred_years_val = years_clf.predict(X_val_y)

years_top1 = accuracy_score(y_val_y, pred_years_val)
years_pm1  = (np.abs(y_val_y - pred_years_val) <= 1).mean()
years_f1   = f1_score(y_val_y, pred_years_val, average="weighted")

metrics_years = pd.DataFrame({
    "metric": ["years_top1_acc","years_pm1_acc","years_f1_weighted"],
    "value": [years_top1, years_pm1, years_f1]
})
display(metrics_years)

feat_names = years_clf.named_steps["prep"].get_feature_names_out()
imp = years_clf.named_steps["clf"].feature_importances_

def aggregate_importances(names, importances):
    agg = {}
    for n, v in zip(names, importances):
        if n.startswith("num__"):
            raw = n.split("num__", 1)[1]
        elif n.startswith("cat__"):
            raw = n.split("cat__", 1)[1].split("_", 1)[0]
        else:
            raw = n
        agg[raw] = agg.get(raw, 0.0) + float(v)
    return pd.Series(agg).sort_values(ascending=False).rename("years_importance")

years_importance = aggregate_importances(feat_names, imp)
display(years_importance.head(25))


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,metric,value
0,years_top1_acc,0.899083
1,years_pm1_acc,0.954128
2,years_f1_weighted,0.879670


OBP_LAST     0.141415
OBP_MEAN5    0.131737
AGE_LAST     0.100948
WAR_LAST     0.100330
AGE_MEAN5    0.099026
SLG_MEAN5    0.094671
WAR_MEAN5    0.094366
SLG_LAST     0.087742
PA_LAST      0.075088
PA_MEAN5     0.074678
PA_STD5      0.000000
WAR_STD5     0.000000
OBP_STD5     0.000000
SLG_STD5     0.000000
AGE_STD5     0.000000
Name: years_importance, dtype: float64

## 6) AAV — Regressor on log scale (HistGradientBoosting)

In [8]:

X_tr_a, X_val_a, y_tr_a, y_val_a = train_test_split(X, y_aav, test_size=0.25, random_state=RANDOM_STATE)

aav_pipe = Pipeline([
    ("prep", preprocessor),
    ("reg", HistGradientBoostingRegressor(
        learning_rate=0.08, max_depth=None, max_bins=255,
        l2_regularization=1.0, min_samples_leaf=20,
        random_state=RANDOM_STATE
    ))
])
aav_reg = TransformedTargetRegressor(
    regressor=aav_pipe,
    func=np.log1p, inverse_func=np.expm1
)
aav_reg.fit(X_tr_a, y_tr_a)

pred_aav_val = np.maximum(aav_reg.predict(X_val_a), 0)

aav_mae  = mean_absolute_error(y_val_a, pred_aav_val)
aav_rmse = mean_squared_error(y_val_a, pred_aav_val, squared=False)
aav_mape = (np.abs((y_val_a - pred_aav_val) / np.where(y_val_a==0, np.nan, y_val_a)))
aav_mape = pd.Series(aav_mape).replace([np.inf, -np.inf], np.nan).dropna().mean()
aav_smape = (np.abs(pred_aav_val - y_val_a) / ((np.abs(y_val_a) + np.abs(pred_aav_val)) / 2.0))
aav_smape = pd.Series(aav_smape).replace([np.inf, -np.inf], np.nan).dropna().mean()

metrics_aav = pd.DataFrame({
    "metric": ["aav_mae","aav_rmse","aav_mape","aav_smape"],
    "value": [aav_mae, aav_rmse, aav_mape, aav_smape]
})
display(metrics_aav)

reg_fitted = aav_reg.regressor_  # fitted Pipeline
feat_names_reg = reg_fitted.named_steps["prep"].get_feature_names_out()
imp_reg = reg_fitted.named_steps["reg"].feature_importances_
aav_importance = pd.Series(imp_reg, index=feat_names_reg)

def aggregate_importances_raw(s: pd.Series):
    agg = {}
    for n, v in s.items():
        if n.startswith("num__"):
            raw = n.split("num__", 1)[1]
        elif n.startswith("cat__"):
            raw = n.split("cat__", 1)[1].split("_", 1)[0]
        else:
            raw = n
        agg[raw] = agg.get(raw, 0.0) + float(v)
    return pd.Series(agg).sort_values(ascending=False)

aav_importance_raw = aggregate_importances_raw(aav_importance).rename("aav_importance")
display(aav_importance_raw.head(25))


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,metric,value
0,aav_mae,5.054569e+06
1,aav_rmse,9.651428e+06
2,aav_mape,1.168976e+00
3,aav_smape,9.860864e-01


AttributeError: 'HistGradientBoostingRegressor' object has no attribute 'feature_importances_'

## 7) Tiered AAV metrics (by actual AAV on validation)

In [9]:

tiers = pd.cut(y_val_a, bins=[-1, 5e6, 15e6, 30e6, np.inf], labels=["<5M","5–15M","15–30M",">=30M"])
rows = []
for t in tiers.cat.categories:
    mask = tiers == t
    if mask.sum() == 0: 
        continue
    mae = mean_absolute_error(y_val_a[mask], pred_aav_val[mask])
    rmse = mean_squared_error(y_val_a[mask], pred_aav_val[mask], squared=False)
    mape = (np.abs((y_val_a[mask] - pred_aav_val[mask]) / np.where(y_val_a[mask]==0, np.nan, y_val_a[mask])))
    mape = pd.Series(mape).replace([np.inf, -np.inf], np.nan).dropna().mean()
    smape = (np.abs(pred_aav_val[mask] - y_val_a[mask]) / ((np.abs(y_val_a[mask]) + np.abs(pred_aav_val[mask])) / 2.0))
    smape = pd.Series(smape).replace([np.inf, -np.inf], np.nan).dropna().mean()
    rows.append({"tier": str(t), "n": int(mask.sum()), "mae": mae, "rmse": rmse, "mape": mape, "smape": smape})
tier_metrics = pd.DataFrame(rows).sort_values("mae", ascending=False)
display(tier_metrics)


,tier,n,mae,rmse,mape,smape
3,>=30M,5,2.445259e+07,3.003183e+07,0.487826,0.758106
2,15–30M,16,1.080742e+07,1.289410e+07,0.504308,0.803844
1,5–15M,23,5.094803e+06,6.043870e+06,0.693173,1.025691
0,<5M,65,2.132092e+06,5.742585e+06,1.689475,1.037642


## 8) AAV Quantile bands (P10/P50/P90) + coverage on validation

In [10]:

def quantile_pipe(alpha):
    return Pipeline([
        ("prep", preprocessor),
        ("reg", GradientBoostingRegressor(loss="quantile", alpha=alpha, random_state=RANDOM_STATE))
    ])

q10 = quantile_pipe(0.10).fit(X_tr_a, y_tr_a)
q50 = quantile_pipe(0.50).fit(X_tr_a, y_tr_a)
q90 = quantile_pipe(0.90).fit(X_tr_a, y_tr_a)

val_p10 = np.maximum(q10.predict(X_val_a), 0)
val_p50 = np.maximum(q50.predict(X_val_a), 0)
val_p90 = np.maximum(q90.predict(X_val_a), 0)

q_stack = np.sort(np.vstack([val_p10, val_p50, val_p90]).T, axis=1)
val_p10, val_p50, val_p90 = q_stack[:,0], q_stack[:,1], q_stack[:,2]

coverage80 = ((y_val_a >= val_p10) & (y_val_a <= val_p90)).mean()
print(f"Validation 80% coverage (between P10 and P90): {coverage80:.3f}")


Validation 80% coverage (between P10 and P90): 0.752


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWar

## 9) Finalize on all 2025 → Score 2026

In [11]:

years_clf_final = Pipeline([("prep", preprocessor), ("clf", years_clf.named_steps["clf"])]).fit(X, y_years)
aav_reg_final = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocessor), ("reg", aav_reg.regressor_.named_steps["reg"])]),
    func=np.log1p, inverse_func=np.expm1
).fit(X, y_aav)

q10_final = quantile_pipe(0.10).fit(X, y_aav)
q50_final = quantile_pipe(0.50).fit(X, y_aav)
q90_final = quantile_pipe(0.90).fit(X, y_aav)

X_26 = X_2026.reindex(columns=X.columns, fill_value=np.nan)
pred26_years = years_clf_final.predict(X_26).clip(1, 12).astype(int)
pred26_aav   = np.maximum(aav_reg_final.predict(X_26), 0)

p10_26 = np.maximum(q10_final.predict(X_26), 0)
p50_26 = np.maximum(q50_final.predict(X_26), 0)
p90_26 = np.maximum(q90_final.predict(X_26), 0)
q_stack26 = np.sort(np.vstack([p10_26, p50_26, p90_26]).T, axis=1)
p10_26, p50_26, p90_26 = q_stack26[:,0], q_stack26[:,1], q_stack26[:,2]

pred26_total = pred26_years * pred26_aav
total_p10 = pred26_years * p10_26
total_p50 = pred26_years * p50_26
total_p90 = pred26_years * p90_26

id_cols_26 = [c for c in ["URL_ID","PLAYER_ID","PLAYER_NAME","NAME"] if c in X_2026.columns]
out_2026 = X_2026[id_cols_26].copy()
out_2026["PRED_YEARS"] = pred26_years
out_2026["PRED_AAV"]   = np.round(pred26_aav, 0)
out_2026["PRED_TOTAL"] = np.round(pred26_total, 0)
out_2026["AAV_P10"] = np.round(p10_26, 0)
out_2026["AAV_P50"] = np.round(p50_26, 0)
out_2026["AAV_P90"] = np.round(p90_26, 0)
out_2026["TOTAL_P10"] = np.round(total_p10, 0)
out_2026["TOTAL_P50"] = np.round(total_p50, 0)
out_2026["TOTAL_P90"] = np.round(total_p90, 0)

display(out_2026.head(10))


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWar

,URL_ID,PLAYER_ID,PLAYER_NAME,PRED_YEARS,PRED_AAV,PRED_TOTAL,AAV_P10,AAV_P50,AAV_P90,TOTAL_P10,TOTAL_P50,TOTAL_P90
0,https://www.baseball-reference.com/players/a/a...,andujmi01,Miguel Andujar,1,7791898.0,7791898.0,974974.0,11253171.0,15481160.0,974974.0,11253171.0,15481160.0
1,https://www.baseball-reference.com/players/a/a...,arciaor01,Orlando Arcia,1,374605.0,374605.0,287841.0,2646167.0,4585600.0,287841.0,2646167.0,4585600.0
2,https://www.baseball-reference.com/players/a/a...,arraelu01,Luis Arraez,1,333550.0,333550.0,630188.0,4637016.0,15260769.0,630188.0,4637016.0,15260769.0
3,https://www.baseball-reference.com/players/b/b...,bichebo01,Bo Bichette,1,9990642.0,9990642.0,106658.0,9363131.0,18248119.0,106658.0,9363131.0,18248119.0
4,https://www.baseball-reference.com/players/c/c...,caratvi01,Victor Caratini,1,3257142.0,3257142.0,974974.0,3868830.0,15260769.0,974974.0,3868830.0,15260769.0
5,https://www.baseball-reference.com/players/c/c...,castrwi01,Willi Castro,1,1753766.0,1753766.0,974974.0,2058073.0,6098505.0,974974.0,2058073.0,6098505.0
6,https://www.baseball-reference.com/players/c/c...,confomi01,Michael Conforto,1,18027494.0,18027494.0,974974.0,3292742.0,9846759.0,974974.0,3292742.0,9846759.0
7,https://www.baseball-reference.com/players/d/d...,dejonpa01,Paul DeJong,1,357105.0,357105.0,287841.0,3303632.0,11361174.0,287841.0,3303632.0,11361174.0
8,https://www.baseball-reference.com/players/f/f...,florewi01,Wilmer Flores,1,31380967.0,31380967.0,974974.0,4193258.0,15260769.0,974974.0,4193258.0,15260769.0
9,https://www.baseball-reference.com/players/f/f...,francty01,Ty France,1,3051435.0,3051435.0,974974.0,2937632.0,14349383.0,974974.0,2937632.0,14349383.0


## 10) Save artifacts

In [14]:
# --- Patch: ensure aav_importance_raw exists and save safely ---

import pandas as pd
import numpy as np

def _aggregate_importances_raw(series_with_feature_space_names: pd.Series) -> pd.Series:
    agg = {}
    for n, v in series_with_feature_space_names.items():
        if n.startswith("num__"):
            raw = n.split("num__", 1)[1]
        elif n.startswith("cat__"):
            raw = n.split("cat__", 1)[1].split("_", 1)[0]
        else:
            raw = n
        agg[raw] = agg.get(raw, 0.0) + float(v)
    return pd.Series(agg).sort_values(ascending=False)

def compute_aav_importance_raw_from_model(fitted_ttr, preprocessor, X_train, y_train):
    """
    Try impurity importances first; if unavailable, fall back to permutation importance.
    """
    # Pull the fitted pipeline from TransformedTargetRegressor
    reg_pipe = fitted_ttr.regressor_                 # Pipeline(prep -> reg)
    prep = reg_pipe.named_steps["prep"]
    reg  = reg_pipe.named_steps["reg"]

    # Attempt impurity-based importances
    if hasattr(reg, "feature_importances_"):
        feat_names = prep.get_feature_names_out()
        s = pd.Series(reg.feature_importances_, index=feat_names)
        return _aggregate_importances_raw(s).rename("aav_importance")

    # Fallback: permutation importance in original feature space
    from sklearn.inspection import permutation_importance
    r = permutation_importance(
        fitted_ttr, X_train, y_train,
        n_repeats=10, random_state=42,
        scoring="neg_mean_absolute_error"
    )
    return pd.Series(r.importances_mean, index=X_train.columns).sort_values(ascending=False).rename("aav_perm_importance")

# If missing, recompute from the FINAL model (trained on all data)
if "aav_importance_raw" not in globals():
    aav_importance_raw = compute_aav_importance_raw_from_model(
        aav_reg_final, preprocessor, X, y_aav
    )

# Also make the years Series save robust
if "years_importance" in globals() and isinstance(years_importance, pd.Series):
    years_importance_df = years_importance.to_frame(name=years_importance.name or "years_importance")
else:
    years_importance_df = None

# Prepare the AAV importance as DataFrame for CSV
aav_importance_df = aav_importance_raw.to_frame(name=aav_importance_raw.name or "aav_importance")
print("Patch ready: aav_importance_raw shape:", aav_importance_df.shape)


art_dir = Path("artifacts"); art_dir.mkdir(exist_ok=True, parents=True)

pred_csv = art_dir / "HTM_2026_predictions_v41.csv"
metrics_csv = art_dir / "HTM_metrics_v41.csv"
tier_csv = art_dir / "HTM_metrics_tiered_v41.csv"
imp_years_csv = art_dir / "HTM_years_importance_v41.csv"
imp_aav_csv = art_dir / "HTM_aav_importance_v41.csv"
years_model_pkl = art_dir / "HTM_years_classifier_v41.joblib"
aav_model_pkl = art_dir / "HTM_aav_regressor_v41.joblib"

metrics_all = pd.concat([metrics_years, metrics_aav], ignore_index=True)

out_2026.to_csv(pred_csv, index=False)
metrics_all.to_csv(metrics_csv, index=False)
tier_metrics.to_csv(tier_csv, index=False)
if years_importance_df is not None:
    years_importance_df.to_csv(imp_years_csv)
aav_importance_df.to_csv(imp_aav_csv)

joblib.dump(years_clf_final, years_model_pkl)
joblib.dump(aav_reg_final, aav_model_pkl)

print("Saved:")
for p in [pred_csv, metrics_csv, tier_csv, imp_years_csv, imp_aav_csv, years_model_pkl, aav_model_pkl]:
    print("-", p)


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWar

/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWar

Patch ready: aav_importance_raw shape: (20, 1)
Saved:
- artifacts/HTM_2026_predictions_v41.csv
- artifacts/HTM_metrics_v41.csv
- artifacts/HTM_metrics_tiered_v41.csv
- artifacts/HTM_years_importance_v41.csv
- artifacts/HTM_aav_importance_v41.csv
- artifacts/HTM_years_classifier_v41.joblib
- artifacts/HTM_aav_regressor_v41.joblib


/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['PA_TREND' 'WAR_TREND' 'OBP_TREND' 'SLG_TREND' 'AGE_TREND']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/bclem213/anaconda3/lib/python3.11/site-packages/sklearn/impute/_base.py:555: UserWar